In [ ]:
import requests
import time
import csv
import os
import datetime

def get_nginx_stats():
    """
    http://localhost:8949/nginx_status 에서 nginx 상태 정보를 받아와서
    active connections, accepts, handled, requests, reading, writing, waiting 값을 추출합니다.
    """
    try:
        res = requests.get("http://localhost:8949/nginx_status")
        res_text = res.text
    except Exception as e:
        print("nginx 상태 정보를 가져오는데 실패:", e)
        return None

    # 줄 단위로 분리
    lines = res_text.strip().split('\n')
    if len(lines) < 4:
        print("응답 형식이 예상과 다릅니다.")
        return None

    try:
        # 첫 번째 줄: "Active connections: 3"
        ac = int(lines[0].split(':')[1].strip())
        accepts = int(lines[2].split()[0].strip())  # accepts 추출
        handled = int(lines[2].split()[1].strip())   # handled 추출
        req_count = int(lines[2].split()[2].strip())   # requests 추출
        reading = int(lines[3].split()[1].strip())     # reading 추출
        writing = int(lines[3].split()[3].strip())     # writing 추출
        waiting = int(lines[3].split()[5].strip())     # waiting 추출
    except Exception as e:
        print("데이터 파싱 중 오류 발생:", e)
        return None

    return ac, accepts, handled, req_count, reading, writing, waiting

# CSV 파일 이름 및 헤더 설정 (총 11 컬럼)
csv_filename = 'nginx_stats.csv'
if not os.path.exists(csv_filename):
    with open(csv_filename, 'w', newline='') as csvfile:
        csvwriter = csv.writer(csvfile)
        header = [
            'timestamp', 
            'active_connections', 
            'cumulative_accepts', 
            'delta_accepts', 
            'cumulative_handled', 
            'delta_handled', 
            'cumulative_requests', 
            'delta_requests', 
            'reading', 
            'writing', 
            'waiting'
        ]
        csvwriter.writerow(header)

# 이전 누적값 저장 변수 (첫 실행 시 차분은 0)
prev_accepts = None
prev_handled = None
prev_req = None

print("10초 간격으로 nginx 상태 정보를 기록합니다. 중단하려면 Ctrl+C를 누르세요.")

while True:
    stats = get_nginx_stats()
    if stats:
        ac, accepts, handled, req_count, reading, writing, waiting = stats
        
        # 첫 실행이면 차분은 0, 이후부터 이전 값과의 차이를 계산
        if prev_accepts is None:
            delta_accepts = 0
            delta_handled = 0
            delta_req = 0
        else:
            delta_accepts = accepts - prev_accepts
            delta_handled = handled - prev_handled
            delta_req = req_count - prev_req
        
        # 현재 값을 이전값으로 저장
        prev_accepts, prev_handled, prev_req = accepts, handled, req_count
        
        # 현재 시각
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # CSV에 저장할 행 (총 11개 컬럼)
        row = [timestamp, ac, accepts, delta_accepts, handled, delta_handled, req_count, delta_req, reading, writing, waiting]
        
        # CSV 파일에 행 추가(append)
        with open(csv_filename, 'a', newline='') as csvfile:
            csvwriter = csv.writer(csvfile)
            csvwriter.writerow(row)
        
        print(f"{timestamp} - 기록: {row}")
    else:
        print("데이터를 기록하지 못했습니다.")

    # 10초 대기
    time.sleep(10)


10초 간격으로 nginx 상태 정보를 기록합니다. 중단하려면 Ctrl+C를 누르세요.
2025-03-06 14:44:55 - 기록: ['2025-03-06 14:44:55', 1, 144, 0, 144, 0, 143, 0, 0, 1, 0]
2025-03-06 14:45:05 - 기록: ['2025-03-06 14:45:05', 1, 147, 3, 147, 3, 146, 3, 0, 1, 0]
2025-03-06 14:45:15 - 기록: ['2025-03-06 14:45:15', 1, 150, 3, 150, 3, 149, 3, 0, 1, 0]
2025-03-06 14:45:25 - 기록: ['2025-03-06 14:45:25', 1, 153, 3, 153, 3, 152, 3, 0, 1, 0]
2025-03-06 14:45:32 - 기록: ['2025-03-06 14:45:32', 1, 156, 3, 156, 3, 155, 3, 0, 1, 0]
2025-03-06 14:45:42 - 기록: ['2025-03-06 14:45:42', 1, 159, 3, 159, 3, 158, 3, 0, 1, 0]
2025-03-06 14:45:52 - 기록: ['2025-03-06 14:45:52', 1, 162, 3, 162, 3, 161, 3, 0, 1, 0]
2025-03-06 14:46:00 - 기록: ['2025-03-06 14:46:00', 1, 165, 3, 165, 3, 164, 3, 0, 1, 0]
2025-03-06 14:46:10 - 기록: ['2025-03-06 14:46:10', 1, 168, 3, 168, 3, 167, 3, 0, 1, 0]
2025-03-06 14:46:20 - 기록: ['2025-03-06 14:46:20', 1, 171, 3, 171, 3, 170, 3, 0, 1, 0]
2025-03-06 14:46:27 - 기록: ['2025-03-06 14:46:27', 1, 174, 3, 174, 3, 173, 3, 0, 1, 0]
2025-

In [1]:
import requests
import time
import csv
import os
import datetime

def get_nginx_stats():
    """
    http://localhost:8949/nginx_status 에서 nginx 상태 정보를 받아와서
    active connections, accepts, handled, requests, reading, writing, waiting 값을 추출
    """
    try:
        res = requests.get("http://localhost:8949/nginx_status")
        res_text = res.text
    except Exception as e:
        print("nginx 상태 정보를 가져오는데 실패:", e)
        return None

    # 줄 단위로 분리
    lines = res_text.strip().split('\n')
    if len(lines) < 4:
        print("응답 형식이 예상과 다릅니다.")
        return None

    try:
        # 첫 번째 줄: "Active connections: 3"
        ac = int(lines[0].split(':')[1].strip())
        accepts = int(lines[2].split()[0].strip()) # accepts 추출
        handled = int(lines[2].split()[1].strip()) # handled 추출
        req_count = int(lines[2].split()[2].strip()) # requests 추출
        reading = int(lines[3].split()[1].strip()) # reading 추출
        writing = int(lines[3].split()[3].strip()) # writing 추출
        waiting = int(lines[3].split()[5].strip()) # writing 추출
        

    except Exception as e:
        print("데이터 파싱 중 오류 발생:", e)
        return None

    return ac, accepts, handled, req_count, reading, writing, waiting

# CSV 파일 이름 및 헤더 설정
csv_filename = 'nginx_stats.csv'
if not os.path.exists(csv_filename):
    with open(csv_filename, 'w', newline='') as csvfile:
        csvwriter = csv.writer(csvfile)
        header = ['timestamp', 'active_connections', 'delta_accepts', 'delta_handled', 'delta_requests', 'reading', 'writing', 'waiting']
        csvwriter.writerow(header)

# 이전 누적값 저장 변수 (첫 실행 시 차분은 0)
prev_accepts = None
prev_handled = None
prev_req = None

print("10초 간격으로 nginx 상태 정보를 기록합니다. 중단하려면 Ctrl+C를 누르세요.")

while True:
    stats = get_nginx_stats()
    if stats:
        ac, accepts, handled, req_count, reading, writing, waiting = stats
        
        # 처음에는 0, 그 후에는 이전 값과의 차이를 계산
        if prev_accepts is None:
            delta_accepts = 0
            delta_handled = 0
            delta_req = 0
        else:
            delta_accepts = accepts - prev_accepts
            delta_handled = handled - prev_handled
            delta_req = req_count - prev_req
        
        # 현재 값을 이전값으로 저장
        prev_accepts, prev_handled, prev_req = accepts, handled, req_count
        
        # 현재 시각
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # CSV에 저장할 행
        row = [timestamp, ac, delta_accepts, delta_handled, delta_req, reading, writing, waiting]
        
        # CSV 파일에 행 추가(append)
        with open(csv_filename, 'a', newline='') as csvfile:
            csvwriter = csv.writer(csvfile)
            csvwriter.writerow(row)
        
        print(f"{timestamp} - 기록: {row}")
    else:
        print("데이터를 기록하지 못했습니다.")

    # 10초 대기
    time.sleep(10)

10초 간격으로 nginx 상태 정보를 기록합니다. 중단하려면 Ctrl+C를 누르세요.
2025-03-06 15:11:54 - 기록: ['2025-03-06 15:11:54', 3, 0, 0, 0, 0, 1, 2]
2025-03-06 15:12:01 - 기록: ['2025-03-06 15:12:01', 3, 480, 480, 480, 0, 1, 2]
2025-03-06 15:12:11 - 기록: ['2025-03-06 15:12:11', 3, 674, 674, 674, 0, 1, 2]
2025-03-06 15:12:21 - 기록: ['2025-03-06 15:12:21', 4, 895, 895, 895, 0, 1, 3]
2025-03-06 15:12:29 - 기록: ['2025-03-06 15:12:29', 3, 1087, 1087, 1087, 0, 1, 2]
2025-03-06 15:12:39 - 기록: ['2025-03-06 15:12:39', 3, 1279, 1279, 1279, 0, 1, 2]
2025-03-06 15:12:49 - 기록: ['2025-03-06 15:12:49', 2, 1479, 1479, 1479, 0, 1, 1]
2025-03-06 15:12:56 - 기록: ['2025-03-06 15:12:56', 1, 1683, 1683, 1683, 0, 1, 0]
2025-03-06 15:13:06 - 기록: ['2025-03-06 15:13:06', 3, 1883, 1883, 1883, 0, 2, 1]
2025-03-06 15:13:16 - 기록: ['2025-03-06 15:13:16', 1, 2066, 2066, 2066, 0, 1, 0]
2025-03-06 15:13:24 - 기록: ['2025-03-06 15:13:24', 1, 2279, 2279, 2279, 0, 1, 0]
2025-03-06 15:13:34 - 기록: ['2025-03-06 15:13:34', 1, 2478, 2478, 2478, 0, 1, 0]
2025-03-0

KeyboardInterrupt: 